# Error Detection 
The objective of this phase is to conduct a comprehensive analysis of the discrepancies identified during the Comparison phase. Rather than simply knowing that errors exist, we now investigate WHY they occurred, categorize them by business impact, and prioritize resolution efforts.

After comparison revealed £17,857.56 in financial variance across 1,980 discrepant records, this phase focuses on transforming raw discrepancy data into actionable business intelligence.

This step includes:

- Error Categorization & Classification:
    - Systematic grouping of discrepancies by type (data quality, system sync, business logic)
    - Financial impact scoring (High/Medium/Low risk categories)
    - Frequency analysis to identify the most common error patterns

- Root Cause Analysis:
    - Investigation of underlying causes for each error category
    - Pattern detection across time periods, products, and customer segments  
    - Correlation analysis between different types of errors

- Business Impact Assessment:
    - Quantification of operational and financial risks
    - Assessment of compliance and audit implications
    - Evaluation of customer experience impact

- Trend & Pattern Analysis:
    - Temporal distribution of errors (seasonal patterns, system events)
    - Product/customer segment analysis (which areas are most affected)
    - Error concentration analysis (are problems systemic or isolated?)

- Prioritization Framework:
    - Development of error priority matrix based on impact vs. effort to resolve
    - Risk scoring methodology for different error types
    - Recommendations for immediate vs. long-term remediation

- Validation Effectiveness Review:
    - Analysis of how well the Validation phase detected different error types
    - Identification of gaps in current validation processes
    - Recommendations for improving future error detection

## Purpose: 
Transform discrepancy identification into strategic remediation planning. This analysis provides the foundation for targeted resolution efforts and process improvements, ensuring that the most critical issues are addressed first while building systematic defenses against future data quality problems.

# 1. Error Categorization and Classification 

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import ast

def load_data(file_path):
    try:
        df = pd.read_csv(file_path, encoding='latin1')
        print(f"File {file_path} loaded correctly using latin1 encoding. Loaded {len(df)} rows.")
    except Exception:
        df = pd.read_csv(file_path, encoding='cp1252')
        print(f"File {file_path} loaded correctly using cp1252 encoding. Loaded {len(df)} rows.")
    
    # parse validation flags
    df['validation_flags'] = df['validation_flags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else []) # parsing strings in the validation_flags column into actual lists
    return df

df_oms = load_data('validation/oms_final_clean.csv')
df_wms = load_data('validation/wms_final_clean.csv')


File validation/oms_final_clean.csv loaded correctly using latin1 encoding. Loaded 541909 rows.
File validation/wms_final_clean.csv loaded correctly using latin1 encoding. Loaded 542409 rows.


**Step 1** Aggregate duplicates

In [33]:
print("STEP 1: Aggregating duplicates for consistent analysis")

# aggregation logic same as in comparison phase 
agg_logic_oms = {
    'UnitPrice': 'max',
    'InvoiceDate': 'first',
    'CustomerID': 'first',
    'Country': 'first',
    'Description': 'first',
    'validation_status': 'first',
    'validation_flags': lambda x: list(set([item for sublist in x for item in sublist]))
}

# define agg logic for wms
agg_logic_wms = agg_logic_oms.copy()

# apply aggregation
df_oms_agg = df_oms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg(agg_logic_oms).reset_index()
df_wms_agg = df_wms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg(agg_logic_wms).reset_index()

print(f"OMS after aggregation: {len(df_oms_agg)} unique business transactions.")
print(f"WMS after aggregation: {len(df_wms_agg)} unique business transactions.")

STEP 1: Aggregating duplicates for consistent analysis
OMS after aggregation: 536478 unique business transactions.
WMS after aggregation: 536478 unique business transactions.


**Step 2** Create matched datasets

In [34]:
df_matched = pd.merge(df_oms_agg, df_wms_agg, 
                      on=['InvoiceNo', 'StockCode', 'Quantity'],
                      how='inner', suffixes=('_oms', '_wms'))

#variance calculatons
df_matched['UnitPrice_variance'] = df_matched['UnitPrice_wms'] - df_matched['UnitPrice_oms']
df_matched['Total_Value_oms'] = df_matched['Quantity'] * df_matched['UnitPrice_oms']
df_matched['Total_Value_wms'] = df_matched['Quantity'] * df_matched['UnitPrice_wms']
df_matched['Total_Value_variance'] = df_matched['Total_Value_wms'] - df_matched['Total_Value_oms']

print(f"Matched records for analysis: {len(df_matched):,}")
print(f"Total financial variance: £{df_matched['Total_Value_variance'].sum():,.2f}")

Matched records for analysis: 536,478
Total financial variance: £-17,857.56


**Step 3** Error categorization adn classification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

# Load data with proper encoding handling
def load_data(file_path):
    try:
        df = pd.read_csv(file_path, encoding='latin1')
        print(f"File {file_path} loaded correctly using latin1 encoding. Loaded {len(df)} rows.")
    except Exception:
        df = pd.read_csv(file_path, encoding='cp1252')
        print(f"File {file_path} loaded correctly using cp1252 encoding. Loaded {len(df)} rows.")
    
    # Parse validation flags
    df['validation_flags'] = df['validation_flags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])
    return df

df_oms = load_data('validation/oms_final_clean.csv')
df_wms = load_data('validation/wms_final_clean.csv')

print("=== ERROR DETECTION & ROOT CAUSE ANALYSIS ===\n")

# STEP 1: AGGREGATE DUPLICATES (same as in Comparison phase)
print("STEP 1: AGGREGATING DUPLICATES FOR CONSISTENT ANALYSIS\n")

# Define aggregation logic for OMS
agg_logic_oms = {
    'UnitPrice': 'max',
    'InvoiceDate': 'first',
    'CustomerID': 'first',
    'Country': 'first',
    'Description': 'first',
    'validation_status': 'first',
    'validation_flags': lambda x: list(set([item for sublist in x for item in sublist]))
}

# Define aggregation logic for WMS
agg_logic_wms = agg_logic_oms.copy()

# Apply aggregation
df_oms_agg = df_oms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg(agg_logic_oms).reset_index()
df_wms_agg = df_wms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg(agg_logic_wms).reset_index()

print(f"OMS after aggregation: {len(df_oms_agg)} unique business transactions")
print(f"WMS after aggregation: {len(df_wms_agg)} unique business transactions")

# STEP 2: CREATE MATCHED DATASET
print(f"\nSTEP 2: CREATING MATCHED DATASET FOR ERROR ANALYSIS\n")

df_matched = pd.merge(df_oms_agg, df_wms_agg, 
                      on=['InvoiceNo', 'StockCode', 'Quantity'],
                      how='inner', suffixes=('_oms', '_wms'))

# Calculate variances
df_matched['UnitPrice_variance'] = df_matched['UnitPrice_wms'] - df_matched['UnitPrice_oms']
df_matched['Total_Value_oms'] = df_matched['Quantity'] * df_matched['UnitPrice_oms']
df_matched['Total_Value_wms'] = df_matched['Quantity'] * df_matched['UnitPrice_wms']
df_matched['Total_Value_variance'] = df_matched['Total_Value_wms'] - df_matched['Total_Value_oms']

print(f"Matched records for analysis: {len(df_matched):,}")
print(f"Total financial variance: £{df_matched['Total_Value_variance'].sum():,.2f}")

# STEP 3: ERROR CATEGORIZATION & CLASSIFICATION
print(f"\n=== ERROR CATEGORIZATION & CLASSIFICATION ===\n")

# 1. SYSTEMATIC GROUPING BY ERROR TYPE
print("1. SYSTEMATIC GROUPING BY ERROR TYPE\n")

def classify_error_type(row):
    """Classify errors into categories based on characteristics"""
    price_var = row['UnitPrice_variance']
    flags_oms = row['validation_flags_oms'] if isinstance(row['validation_flags_oms'], list) else []
    flags_wms = row['validation_flags_wms'] if isinstance(row['validation_flags_wms'], list) else []
    
    # Data Quality Errors - Missing or corrupted data
    if 'Missing_UnitPrice' in flags_wms or price_var <= -1.0:
        return 'Data Quality Error'
    elif 'Invalid_Date' in flags_wms:
        return 'Data Quality Error'
    
    # System Sync Errors - Duplicate records after aggregation
    elif 'Duplicate_Record' in flags_oms or 'Duplicate_Record' in flags_wms:
        return 'System Sync Error'
    
    # Business Logic Errors - Small price differences (rounding, etc.)
    elif 0 < abs(price_var) <= 0.05:
        return 'Business Logic Error'
    
    # Large unexplained variances
    elif abs(price_var) > 0.05:
        return 'Unknown Error'
    
    # No error
    else:
        return 'No Error'

# Apply classification
df_matched['error_type'] = df_matched.apply(classify_error_type, axis=1)

# Count by error type
error_type_counts = df_matched['error_type'].value_counts()
print("Error Distribution by Type:")
print(error_type_counts)
print(f"\nTotal records analyzed: {len(df_matched):,}")

# 2. FINANCIAL IMPACT SCORING
print(f"\n2. FINANCIAL IMPACT SCORING\n")

def classify_financial_impact(variance):
    """Classify financial impact as High/Medium/Low"""
    abs_variance = abs(variance)
    if abs_variance >= 10.0:
        return 'High Risk'
    elif abs_variance >= 1.0:
        return 'Medium Risk'
    elif abs_variance > 0:
        return 'Low Risk'
    else:
        return 'No Risk'

# Apply financial impact classification
df_matched['financial_impact'] = df_matched['Total_Value_variance'].apply(classify_financial_impact)

# Financial impact analysis
financial_impact_summary = df_matched.groupby('financial_impact').agg({
    'Total_Value_variance': ['count', 'sum', 'mean'],
    'UnitPrice_variance': ['min', 'max']
}).round(2)

print("Financial Impact Classification:")
print(financial_impact_summary)

# 3. ERROR TYPE vs FINANCIAL IMPACT CROSS-ANALYSIS
print(f"\n3. ERROR TYPE vs FINANCIAL IMPACT CROSS-ANALYSIS\n")
cross_analysis = pd.crosstab(df_matched['error_type'], 
                            df_matched['financial_impact'], 
                            margins=True)
print(cross_analysis)

# 4. FREQUENCY ANALYSIS - MOST COMMON ERROR PATTERNS
print(f"\n4. FREQUENCY ANALYSIS - MOST COMMON ERROR PATTERNS\n")

# Filter only records with errors
error_records = df_matched[df_matched['error_type'] != 'No Error']

# A. Most affected products (StockCode)
print("A. TOP 10 MOST AFFECTED PRODUCTS:")
if len(error_records) > 0:
    product_errors = error_records.groupby('StockCode').agg({
        'Total_Value_variance': ['count', 'sum'],
        'error_type': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Mixed'
    }).round(2)
    
    product_errors.columns = ['Error_Count', 'Total_Financial_Impact', 'Most_Common_Error_Type']
    product_errors = product_errors.sort_values('Total_Financial_Impact', key=abs, ascending=False)
    print(product_errors.head(10))
else:
    print("No error records found for product analysis")

# B. Most affected invoices
print(f"\nB. TOP 10 MOST AFFECTED INVOICES:")
if len(error_records) > 0:
    invoice_errors = error_records.groupby('InvoiceNo').agg({
        'Total_Value_variance': ['count', 'sum'],
        'error_type': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Mixed'
    }).round(2)
    
    invoice_errors.columns = ['Error_Count', 'Total_Financial_Impact', 'Most_Common_Error_Type']
    invoice_errors = invoice_errors.sort_values('Total_Financial_Impact', key=abs, ascending=False)
    print(invoice_errors.head(10))
else:
    print("No error records found for invoice analysis")

# C. Error concentration by country
print(f"\nC. ERROR DISTRIBUTION BY COUNTRY:")
if len(error_records) > 0:
    country_errors = error_records.groupby('Country_oms').agg({
        'Total_Value_variance': ['count', 'sum', 'mean'],
        'error_type': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Mixed'
    }).round(2)
    
    country_errors.columns = ['Error_Count', 'Total_Financial_Impact', 'Avg_Error_Value', 'Most_Common_Error_Type']
    country_errors = country_errors.sort_values('Total_Financial_Impact', key=abs, ascending=False)
    print(country_errors.head())
else:
    print("No error records found for country analysis")

# 5. DETAILED ERROR BREAKDOWN BY TYPE
print(f"\n5. DETAILED ERROR BREAKDOWN BY TYPE\n")

for error_type in error_type_counts.index:
    if error_type != 'No Error':
        subset = df_matched[df_matched['error_type'] == error_type]
        financial_impact = subset['Total_Value_variance'].sum()
        avg_impact = subset['Total_Value_variance'].mean()
        
        print(f"{error_type}:")
        print(f"  Records: {len(subset):,}")
        print(f"  Total Financial Impact: £{financial_impact:,.2f}")
        print(f"  Average Impact per Record: £{avg_impact:.2f}")
        print(f"  Price Variance Range: £{subset['UnitPrice_variance'].min():.2f} to £{subset['UnitPrice_variance'].max():.2f}")
        
        # Show validation flags for this error type
        all_flags = []
        for flags in subset['validation_flags_oms'].tolist() + subset['validation_flags_wms'].tolist():
            if isinstance(flags, list):
                all_flags.extend(flags)
        
        if all_flags:
            flag_counts = pd.Series(all_flags).value_counts()
            print(f"  Associated Validation Flags: {dict(flag_counts)}")
        print()

# 6. SUMMARY STATISTICS
print(f"\n=== CLASSIFICATION SUMMARY ===\n")

total_errors = len(df_matched[df_matched['error_type'] != 'No Error'])
total_records = len(df_matched)
error_rate = (total_errors / total_records) * 100

print(f"Overall Error Statistics:")
print(f"  Total records processed: {total_records:,}")
print(f"  Records with errors: {total_errors:,}")
print(f"  Error rate: {error_rate:.2f}%")
print(f"  Total financial exposure: £{df_matched['Total_Value_variance'].sum():,.2f}")

print(f"\nError Type Breakdown:")
for error_type, count in error_type_counts.items():
    if error_type != 'No Error':
        percentage = (count / total_records) * 100
        financial_impact = df_matched[df_matched['error_type'] == error_type]['Total_Value_variance'].sum()
        print(f"  {error_type}: {count:,} records ({percentage:.2f}%) - £{financial_impact:,.2f}")

print(f"\nFinancial Risk Distribution:")
for risk_level in ['High Risk', 'Medium Risk', 'Low Risk']:
    if risk_level in df_matched['financial_impact'].values:
        count = (df_matched['financial_impact'] == risk_level).sum()
        percentage = (count / total_records) * 100
        financial_impact = df_matched[df_matched['financial_impact'] == risk_level]['Total_Value_variance'].sum()
        print(f"  {risk_level}: {count:,} records ({percentage:.2f}%) - £{financial_impact:,.2f}")

# 7. VALIDATION EFFECTIVENESS ANALYSIS
print(f"\n=== VALIDATION EFFECTIVENESS ANALYSIS ===\n")

# Check how many errors were caught by validation flags vs missed
errors_with_flags = 0
errors_without_flags = 0

for idx, row in df_matched[df_matched['error_type'] != 'No Error'].iterrows():
    flags_oms = row['validation_flags_oms'] if isinstance(row['validation_flags_oms'], list) else []
    flags_wms = row['validation_flags_wms'] if isinstance(row['validation_flags_wms'], list) else []
    
    if len(flags_oms) > 0 or len(flags_wms) > 0:
        errors_with_flags += 1
    else:
        errors_without_flags += 1

if total_errors > 0:
    detection_rate = (errors_with_flags / total_errors) * 100
    print(f"Validation Detection Effectiveness:")
    print(f"  Errors detected by validation flags: {errors_with_flags:,} ({detection_rate:.1f}%)")
    print(f"  Errors missed by validation: {errors_without_flags:,} ({100-detection_rate:.1f}%)")
    print(f"  Overall validation effectiveness: {detection_rate:.1f}%")
else:
    print("No errors found to analyze validation effectiveness")


Error Categorization and Classification
Error Distribution by Type:
error_type
No Error                529634
System Sync Error         4879
Business Logic Error       983
Data Quality Error         772
Unknown Error              210
Name: count, dtype: int64
Total records analyzed: 536,478
Financial Impact Scoring

Financial Impact Classification:
                 Total_Value_variance                  UnitPrice_variance  \
                                count       sum   mean                min   
financial_impact                                                            
High Risk                         477 -15906.28 -33.35            -550.64   
Low Risk                         1030     46.20   0.04              -0.95   
Medium Risk                       473  -1997.48  -4.22              -9.95   
No Risk                        534498      0.00   0.00               0.00   

                        
                   max  
financial_impact        
High Risk        -0.21  
Low Risk 